# Taiwanese Company Bankruptcy Prediction (imbalanced classification)

Predict whether a company goes bankrupt from 95 financial ratios, where only about 3% of companies are bankrupt.

| | |
|---|---|
| **Task** | Binary classification, highly imbalanced |
| **Data** | UCI dataset #572: 6,819 companies, 95 features, 220 bankrupt (3.2%) |
| **Model** | Decision Tree (entropy, `max_depth=5`, `max_features=20`, `ccp_alpha=0.009`, `class_weight={0: 4, 1: 25}`) |
| **Result** | **69% recall** on bankrupt companies at **29% precision** (test) |

## Approach
- Stratified 75/25 split (`random_state=13`) to keep the bankrupt share equal in train and test.
- A shallow, pruned tree with heavy class weights on the bankrupt class so the model doesn't just predict "not bankrupt".

## Results (test set, 1,705 companies)

| | Predicted healthy | Predicted bankrupt |
|---|---:|---:|
| **Actually healthy** | 1,556 | 94 |
| **Actually bankrupt** | 17 | 38 |

- Accuracy is 93.5%, but that number is misleading: always predicting "healthy" would score 96.8%.
- Bankrupt class: precision 0.29, recall 0.69, F1 0.41. The model catches 38 of 55 bankruptcies at the cost of 94 false alarms.
- Train recall was 0.87, so the model generalizes worse on the minority class.

## Limitations & next steps
- Hyperparameters were set by hand (the notebook has scratch notes with other values) without a separate validation set, so the test score is probably a bit optimistic. Use cross-validated search instead.
- Evaluate with PR-AUC / ROC-AUC and choose the decision threshold on purpose, based on the cost of a missed bankruptcy vs. a false alarm.
- Try ensemble models and resampling (SMOTE) and compare with this baseline.


In [1]:
!pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression


In [3]:

# fetch dataset
taiwanese_bankruptcy_prediction = fetch_ucirepo(id=572)

# data (as pandas dataframes)
X = taiwanese_bankruptcy_prediction.data.features
y = taiwanese_bankruptcy_prediction.data.targets

# metadata
print(taiwanese_bankruptcy_prediction.metadata)

# variable information
print(taiwanese_bankruptcy_prediction.variables)


{'uci_id': 572, 'name': 'Taiwanese Bankruptcy Prediction', 'repository_url': 'https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction', 'data_url': 'https://archive.ics.uci.edu/static/public/572/data.csv', 'abstract': 'The data were collected from the Taiwan Economic Journal  for the years 1999 to 2009. Company bankruptcy was defined based on the business regulations of the Taiwan Stock Exchange.', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 6819, 'num_features': 95, 'feature_types': ['Integer'], 'demographics': [], 'target_col': ['Bankrupt?'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Fri Mar 15 2024', 'dataset_doi': '10.24432/C5004D', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': None, 'purpose': None, 'funded_by': None, 'instances_represent': None, 'recommended_data_splits': None, 'sensitive_data': Non

In [4]:
df = pd.concat([X,y],axis=1)
df


,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability,Bankrupt?
0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,...,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469,1
1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,0.781506,...,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794,1
2,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,...,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474,1
3,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,0.781241,...,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982,1
4,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,0.781550,...,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6814,0.493687,0.539468,0.543230,0.604455,0.604462,0.998992,0.797409,0.809331,0.303510,0.781588,...,0.000466,0.623620,0.604455,0.840359,0.279606,0.027064,0.566193,1,0.029890,0
6815,0.475162,0.538269,0.524172,0.598308,0.598308,0.998992,0.797414,0.809327,0.303520,0.781586,...,0.001959,0.623931,0.598306,0.840306,0.278132,0.027009,0.566018,1,0.038284,0
6816,0.472725,0.533744,0.520638,0.610444,0.610213,0.998984,0.797401,0.809317,0.303512,0.781546,...,0.002840,0.624156,0.610441,0.840138,0.275789,0.026791,0.565158,1,0.097649,0
6817,0.506264,0.559911,0.554045,0.607850,0.607850,0.999074,0.797500,0.809399,0.303498,0.781663,...,0.002837,0.623957,0.607846,0.841084,0.277547,0.026822,0.565302,1,0.044009,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6819 entries, 0 to 6818
Data columns (total 96 columns):
 #   Column                                                    Non-Null Count  Dtype  
---  ------                                                    --------------  -----  
 0    ROA(C) before interest and depreciation before interest  6819 non-null   float64
 1    ROA(A) before interest and % after tax                   6819 non-null   float64
 2    ROA(B) before interest and depreciation after tax        6819 non-null   float64
 3    Operating Gross Margin                                   6819 non-null   float64
 4    Realized Sales Gross Margin                              6819 non-null   float64
 5    Operating Profit Rate                                    6819 non-null   float64
 6    Pre-tax net Interest Rate                                6819 non-null   float64
 7    After-tax net Interest Rate                              6819 non-null   float64
 8    Non-industry inco

In [6]:
df['Bankrupt?'].value_counts()

,count
Bankrupt?,
0,6599
1,220


In [7]:
df.corr()['Bankrupt?'].sort_values(ascending=False)

,Bankrupt?
Bankrupt?,1.000000
Debt ratio %,0.250161
Current Liability to Assets,0.194494
Borrowing dependency,0.176543
Current Liability to Current Assets,0.171306
...,...
ROA(C) before interest and depreciation before interest,-0.260807
ROA(B) before interest and depreciation after tax,-0.273051
ROA(A) before interest and % after tax,-0.282941
Net Income to Total Assets,-0.315457


In [8]:
x = df.drop('Bankrupt?',axis=1)
y = df['Bankrupt?']

In [9]:
x

,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,0.781506,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,0.781241,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,0.781550,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6814,0.493687,0.539468,0.543230,0.604455,0.604462,0.998992,0.797409,0.809331,0.303510,0.781588,...,0.799927,0.000466,0.623620,0.604455,0.840359,0.279606,0.027064,0.566193,1,0.029890
6815,0.475162,0.538269,0.524172,0.598308,0.598308,0.998992,0.797414,0.809327,0.303520,0.781586,...,0.799748,0.001959,0.623931,0.598306,0.840306,0.278132,0.027009,0.566018,1,0.038284
6816,0.472725,0.533744,0.520638,0.610444,0.610213,0.998984,0.797401,0.809317,0.303512,0.781546,...,0.797778,0.002840,0.624156,0.610441,0.840138,0.275789,0.026791,0.565158,1,0.097649
6817,0.506264,0.559911,0.554045,0.607850,0.607850,0.999074,0.797500,0.809399,0.303498,0.781663,...,0.811808,0.002837,0.623957,0.607846,0.841084,0.277547,0.026822,0.565302,1,0.044009


In [10]:
y

,Bankrupt?
0,1
1,1
2,1
3,1
4,1
...,...
6814,0
6815,0
6816,0
6817,0


In [11]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=13,stratify=y)

In [12]:
model = DecisionTreeClassifier(class_weight={0:4,1:25},ccp_alpha=0.009,random_state=13,criterion='entropy',max_depth=5,max_features=20)
model.fit(x_train,y_train)

DecisionTreeClassifier(ccp_alpha=0.009, class_weight={0: 4, 1: 25},
                       criterion='entropy', max_depth=5, max_features=20,
                       random_state=13)

In [13]:
y_pred_train = model.predict(x_train)
y_pred_test = model.predict(x_test)



class_weight={0:4,1:21},ccp_alpha=0.04
                               ,max_depth=5,criterion='log_loss',max_features=20,min_impurity_decrease=0.009,random_state=42,min_weight_fraction_leaf=0.021

                               

In [14]:
print('accuracy for train',accuracy_score(y_train,y_pred_train))
print('classifcaition report for train',classification_report(y_train,y_pred_train))
print('confusion matrix for train',confusion_matrix(y_train,y_pred_train))

accuracy for train 0.9284317559640204
classifcaition report for train               precision    recall  f1-score   support

           0       1.00      0.93      0.96      4949
           1       0.29      0.87      0.44       165

    accuracy                           0.93      5114
   macro avg       0.64      0.90      0.70      5114
weighted avg       0.97      0.93      0.94      5114

confusion matrix for train [[4604  345]
 [  21  144]]


In [15]:
print('accuracy for test',accuracy_score(y_test,y_pred_test))
print('classifcaition report for test',classification_report(y_test,y_pred_test))
print('confusion matrix for test',confusion_matrix(y_test,y_pred_test))

accuracy for test 0.9348973607038124
classifcaition report for test               precision    recall  f1-score   support

           0       0.99      0.94      0.97      1650
           1       0.29      0.69      0.41        55

    accuracy                           0.93      1705
   macro avg       0.64      0.82      0.69      1705
weighted avg       0.97      0.93      0.95      1705

confusion matrix for test [[1556   94]
 [  17   38]]
